In [1]:
import sys
import rqdatac

print("Python：", sys.version)
print("当前 rqdatac：", rqdatac.__version__)

Python： 3.6.10 |Anaconda, Inc.| (default, Jan  7 2020, 21:14:29) 
[GCC 7.3.0]
当前 rqdatac： 3.4.2


In [3]:
import sys

!{sys.executable} -m pip install --user --upgrade --no-cache-dir -i https://pypi.org/simple "rqdatac==3.4.2"

     |████████████████████████████████| 582 kB 492 kB/s eta 0:00:01
  Attempting uninstall: rqdatac
    Found existing installation: rqdatac 3.0.15
    Uninstalling rqdatac-3.0.15:
      Successfully uninstalled rqdatac-3.0.15


# 最大异常日收益率（ABNRETD）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNRETD} = \max_{t} \left| r_{i,t} - \bar{r}_t \right|
$$

其中：

$$
r_{i,t} \text{ 为股票 } i \text{ 在交易日 } t \text{ 的收益率}
$$$$
\bar{r}_t \text{ 为交易日 } t \text{ 的全市场收益率}
$$$$
m \text{ 为交易日窗口，一般取为月度}
$$

## 因子说明

最大异常日收益率（ABNRETD）是每月内所有交易日中，个股收益率与市场收益率之差的绝对值的最大值，属于**极端日收益类因子**。极端收益（无论是大幅上涨还是大幅下跌）的股票，相比收益平稳的股票更能吸引投资者的关注。因子取值越高，表明该股票在当月曾出现过极端的日收益，投资者的关注度也就越高。**然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票**。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度收盘价

# CSSD 模型（横截面收益标准差模型）

**用于检验羊群效应的行为金融模型**

$$
\text{CSSD} = \sqrt{\frac{\sum_{i=1}^{N} (r_i - r_m)^2}{N - 1}}
$$

其中：

$$
r_i \text{ 为个股 } i \text{ 的收益率}
$$$$
r_m \text{ 为市场组合的收益率}
$$$$
N \text{ 为样本中的股票数量}
$$

以 CSSD 作为被解释变量，以两个虚拟变量 $ D_t^u $ 和 $ D_t^d $ 作为解释变量，进行如下时序回归：

$$
\text{CSSD}_t = \alpha + \beta_1 D_t^u + \beta_2 D_t^d + \varepsilon_t
$$

其中：

$$
D_t^u = \begin{cases}1, & \text{若市场收益 } r_{m,t} \text{ 高于给定的上涨阈值} \\0, & \text{否则}\end{cases}
$$$$
D_t^d = \begin{cases}1, & \text{若市场收益 } r_{m,t} \text{ 低于给定的下跌阈值} \\0, & \text{否则}\end{cases}
$$

## 模型说明

CSSD 模型对羊群效应的检验，主要依据时序回归中市场极端收益项（即 $ D_t^u $ 和 $ D_t^d $）的回归系数 $ \beta_1 $ 和 $ \beta_2 $ 的符号及显著性。当 $ \beta_1 $ 或 $ \beta_2 $ 显著小于 0 时，表明在市场大幅上涨或大幅下跌的极端行情下，横截面收益标准差（CSSD）显著下降，意味着个股收益趋于一致，投资者的买卖行为趋向同步，从而反映出市场中存在明显的羊群效应。

数据：日度收盘价、日度可交易股票数量

# CSAD 模型（横截面绝对偏差模型）

**用于检验羊群效应的行为金融模型**

利用个股收益 $r_i$ 和市场收益 $R_m$ 计算股票收益的横截面绝对偏差：

$$
\text{CSAD} = \frac{\sum_{i=1}^{N} |r_i - R_m|}{N}
$$

其中：

* $r_i$ 为个股 $i$ 的收益率；
* $R_m$ 为市场组合的收益率；
* $N$ 为样本中的股票数量。

以 CSAD 作为被解释变量，以市场收益的绝对值和市场收益的平方作为解释变量，进行如下时序回归：

$$
\text{CSAD}_t = \alpha + \beta_1 |R_{mt}| + \beta_2 R_{mt}^2 + \varepsilon_t
$$

## 模型说明

CSAD 模型对羊群效应的检验，主要依据时序回归中市场收益平方项（$R_{mt}^2$）前的回归系数 $\beta_2$ 的符号及显著性。当 $\beta_2$ 显著小于 0 时，表明在市场大幅上涨或大幅下跌的极端行情下，横截面绝对偏差（CSAD）显著下降，意味着个股收益趋于一致，投资者的买卖行为趋向同步，从而反映出市场中存在明显的羊群效应。当 $\beta_1$ 和 $\beta_2$ 均显著小于 0 时，说明市场羊群效应非常显著，投资者在极端行情下的从众行为尤为突出。

数据：日度收盘价、日度可交易股票数量

# CSAD 模型（基于“小型”板块）

**用于构建羊群效应因子的行为金融模型**

① 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_t$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；

② 选取与 $S_t$ 相关系数最大的 9 只股票，连同 $S_t$ 一起构成一个包含 10 只股票的“小型”板块：

$$
S_i,\quad i = 1, 2, \dots, 10
$$

③ 计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - \bar{r}_t \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$\bar{r}_t$ 为板块内所有股票在交易日 $t$ 的平均收益率。

④ 对 CSAD 进行时序标准化，并计算过去 20 个交易日的均值，作为羊群效应因子：

$$
Z_{jt} = \frac{\text{CSAD}_{jt} - \mu_{\text{CSAD}_j}}{\sigma_{\text{CSAD}_j}}
$$$$
f_{jt} = -\frac{1}{20} \sum_{k=1}^{20} Z_{j,t-k+1}
$$

其中，$j$ 表示第 $j$ 个“小型”板块，$\mu_{\text{CSAD}_j}$ 和 $\sigma_{\text{CSAD}_j}$ 分别为该板块 CSAD 在样本期内的均值和标准差，$Z_{jt}$ 为标准化后的 CSAD，$f_{jt}$ 为最终的羊群效应因子。

## 因子逻辑说明

CSAD 利用股票收益在横截面上的分散度来衡量市场羊群效应的强弱。为使因子对个股具备选股能力，该模型将与目标个股走势最为相近的股票组成“小型”板块，借助该小板块上的羊群效应强弱来近似反映该股票近期的关注度变化。

因子值 $f_{jt}$ 越大（由于负号处理，对应近期 CSAD 显著偏低），表明板块近期的羊群效应越强，板块内个股收益趋于一致，该板块受市场关注的程度提升越明显，未来收益预期也越高。

数据：日度收盘价

# CSAD 模型（基于板块中地位）

**用于构建羊群效应因子的行为金融模型**

① 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_*$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；

② 选取与 $S_*$ 相关系数最大的 9 只股票，连同 $S_*$ 一起构成一个包含 10 只股票的“小型”板块：

$$
S_i, \quad i = 1, 2, \dots, 10
$$

③ 以 $S_*$ 为中心股票，计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - r_{*,t} \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$r_{*,t}$ 为中心股票 $S_*$ 在交易日 $t$ 的收益率。

④ 对 CSAD 进行时序标准化，并计算过去 20 个交易日的均值，作为羊群效应因子：

$$
Z_{jt} = \frac{\text{CSAD}_{jt} - \mu_{\text{CSAD}_j}}{\sigma_{\text{CSAD}_j}}
$$$$
f_{jt} = -\frac{1}{20} \sum_{k=1}^{20} Z_{j,t-k+1}
$$

其中，$j$ 表示第 $j$ 个“小型”板块，$\mu_{\text{CSAD}_j}$ 和 $\sigma_{\text{CSAD}_j}$ 分别为该板块 CSAD 在样本期内的均值和标准差，$Z_{jt}$ 为标准化后的 CSAD，$f_{jt}$ 为最终的羊群效应因子。

## 因子逻辑说明

CSAD 利用股票收益在横截面上的分散度来衡量市场羊群效应的强弱。为使因子对个股具备选股能力，该模型将与目标个股走势最为相近的股票组成“小型”板块，借助该小板块上的羊群效应强弱来近似反映该股票近期的关注度变化。

与基于板块均值的 CSAD 模型不同，本模型以中心股票 $S_*$ 的收益为参照基准，计算板块内其他股票相对于该中心股票的偏离程度。因子值 $f_{jt}$ 越大（由于负号处理，对应近期 CSAD 显著偏低），表明中心股票收益越接近板块内其他股票的中位数水平，板块近期的羊群效应越强，该板块受市场关注的程度提升越明显，未来收益预期也越高。

数据：日度收盘价

# CSAD 模型（基于时序回归变形）

**用于构建羊群效应因子的行为金融模型**

* 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_*$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；
  
* 选取与 $S_*$ 相关系数最大的 9 只股票，连同 $S_*$ 一起构成一个包含 10 只股票的"小型"板块：
  

$$
S_i, \quad i = 1, 2, \dots, 10
$$

* 以 $S_*$ 为中心股票，计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - r_{*,t} \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$r_{*,t}$ 为中心股票 $S_*$ 在交易日 $t$ 的收益率。

* 以 CSAD 波动率相对于个股波动率的比值变化作为羊群效应因子，分别计算短期比值和长期比值：

$$
\text{ratio}[t_1, t_2] = \frac{\sigma[t_1, t_2](\text{CSAD})}{\sigma[t_1, t_2](r_*)}
$$

其中，$\sigma[t_1, t_2](\text{CSAD})$ 为 CSAD 在时间段 $[t_1, t_2]$ 内的波动率（标准差），$\sigma[t_1, t_2](r_*)$ 为中心股票收益率在同一时间段内的波动率。

最终的羊群效应因子定义为近期比值与长期比值的比率：

$$
f_{jt} = \frac{\text{ratio}[t - 19, t]}{\text{ratio}[t - 119, t]}
$$

其中，分子 $\text{ratio}[t - 19, t]$ 为过去 20 个交易日的短期 CSAD 相对波动率比值，分母 $\text{ratio}[t - 119, t]$ 为过去 120 个交易日的长期 CSAD 相对波动率比值。

## 因子逻辑说明

标准的 CSAD 时序回归模型通过回归系数 $\beta_2$ 的显著性来检验羊群效应（$\beta_2$ 显著为负表明羊群效应存在），核心逻辑是：当羊群效应存在时，股票收益率的分散度（CSAD）随市场极端行情而减小的幅度，大于市场收益水平本身的变化幅度。

考虑到时序长度不足和滞后误差等因素的影响，本模型对标准时序回归方法进行了变形处理。通过比较 CSAD 波动率与中心股票波动率的比值在短期和长期的变化情况，避免了直接进行时序回归可能带来的估计偏差。当 $f_{jt} < 1$ 时，表明近期 CSAD 的相对波动率较长期水平有所下降，即板块内个股收益趋于一致，羊群效应增强，该板块受关注程度提升；反之，当 $f_{jt} > 1$ 时，表明羊群效应减弱。

# 平均异常日收益率（ABNRETAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNRETAVG} = \frac{1}{m} \sum_{t}^{t-m+1} \left( r_{i,t} - \bar{r}_t \right)^2
$$

其中：

* $r_{i,t}$ 为股票 $i$ 在交易日 $t$ 的收益率；
* $\bar{r}_t$ 为交易日 $t$ 的全市场收益率；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

平均异常日收益率（ABNRETAVG）为每月日频收益率与市场收益率差值的均方，属于**极端日收益类因子**。极端收益（无论是大幅上涨还是大幅下跌）的股票，相比收益平稳的股票更能吸引投资者的关注。因子取值越高，表明该股票在当月曾出现过较为极端的日收益偏离，投资者的关注度也就越高。

然而，**投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为**。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度收盘价

# 每日最大异常成交量（ABNVOLD）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNVOLD} = \max_{t} \left( \frac{VOL_{i,t}}{\overline{VOL}_{i,t}} \right)
$$

其中：

* $VOL_{i,t}$ 为股票 $i$ 在交易日 $t$ 的成交量；
* $\overline{VOL}_{i,t}$ 为股票 $i$ 在交易日 $t$ 滚动 1 年的平均成交量；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

每日最大异常成交量（ABNVOLD）为日频成交量除以过去一年平均成交量的最大值，属于**极端交易量类因子**。异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票在当月曾出现过异常放量，投资者的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：成交量

# 每日平均异常成交（ABNVOLAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNVOLAVG} = \frac{1}{m} \sum_{t=1}^{m} \left( \frac{VOL_{i,t}}{\overline{VOL}_{i,t}} \right)
$$

其中：

* $VOL_{i,t}$ 为股票 $i$ 在交易日 $t$ 的成交量；
* $\overline{VOL}_{i,t}$ 为股票 $i$ 在交易日 $t$ 滚动 1 年的平均成交量；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

每日平均异常成交（ABNVOLAVG）为日频成交量除以过去一年平均成交量的均值，属于**极端交易量类因子**。异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票在当月平均异常成交水平越高，投资者的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度成交量

# 未实现盈利量（CGO）

**行为金融因子 —— 投资者参考点与盈亏状态**

当前未实现盈利量定义为当前价格相对于参考价格的相对偏离程度：

$$
\text{CGO}_t = \frac{P_t - RP_t}{RP_t}
$$

其中，参考价格 $RP_t$ 采用换手率加权的历史价格进行构造：

$$
RP_t = \frac{1}{k} \sum_{n=1}^{N} V_{t-n} \prod_{\tau=1}^{n-1} \left( 1 - V_{t-\tau} \right) P_{t-n}
$$

各变量定义如下：

* $P_t$ 为股票在 $t$ 时刻的价格；
* $P_{t-n}$ 为股票在 $t-n$ 时刻的价格；
* $V_t$ 为股票在 $t$ 时刻的换手率；
* $V_{t-n}$ 为股票在 $t-n$ 时刻的换手率；
* $\frac{1}{k}$ 为权重归一化参数，确保所有权重之和为 1；
* $N$ 为截断时间长度，通常可取 $N = 40$ 或 $60$ 个交易日。

## 因子说明

未实现盈利量（Capital Gain Overhang，CGO）衡量了当前投资者整体的盈亏水平，其核心逻辑在于通过比较当前价格与参考价格（即投资者的平均持仓成本）来判断市场参与者的盈利或亏损状态。

该因子通常与股票未来收益呈正相关关系：

* 当投资者处于**盈利状态**（$\text{CGO} > 0$）时，由于厌恶风险，他们更倾向于卖出股票以锁定利润，从而形成卖盘压力，导致股价被低估，未来预期收益上升；
* 当投资者处于**亏损状态**（$\text{CGO} < 0$）时，由于偏好风险（即“赌徒心态”或“处置效应”），他们更倾向于继续持有股票而非止损卖出，导致股价被高估，未来预期收益下降。

长期来看，上述现象在 A 股市场是成立的。但在短期内，浮盈股票面临的卖盘压力可能大于其被低估的幅度，从而出现浮盈股票未来表现反而更差的情况。

数据：日度收盘价、日度换手率

# 衰减加权月频成交量（ATTN）

**行为金融因子 —— 投资者注意力**

衰减加权月频成交量因子通过对过去 12 个月的成交量按月线性衰减加权求和得到，赋予近期成交量更高的权重：

$$
\text{ATTN}_t = \sum_{m=1}^{11} \frac{12 - m}{\sum_{k=1}^{11} k} \cdot VOL_{i, t-m}
$$

其中，分母为权重归一化常数：

$$
\sum_{k=1}^{11} k = \frac{11 \times 12}{2} = 66
$$

因此，因子可简化为：

$$
\text{ATTN}_t = \sum_{m=1}^{11} \frac{12 - m}{66} \cdot VOL_{i, t-m}
$$

各变量定义如下：

* $VOL_{i, t-m}$ 为股票 $i$ 在 $t-m$ 月的月频成交量（共取过去 11 个月，加上当月共覆盖 12 个月度窗口）；
* $m$ 为月份滞后阶数，从 1 到 11，取值越大，对应的成交量越早，权重越低；
* $\frac{12 - m}{66}$ 为线性衰减权重，权重随 $m$ 增大而递减（即近期权重高、远期权重低）。

## 因子说明

衰减加权月频成交量因子通过对股票历史成交量进行时间衰减加权得到，属于**极端交易量类因子**。

异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票近期成交量持续活跃，投资者的关注度也就越高。

然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度成交量

# 平均换手率（TURNAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{TURNAVG} = \frac{1}{m} \sum_{t=1}^{m} \text{TURN}_{i,t}
$$

其中：

* $\text{TURN}_{i,t}$ 为股票 $i$ 在交易日 $t$ 的换手率；
* $m$ 为交易日窗口，一般取为月度（即当月所有交易日）。

## 因子说明

平均换手率（TURNAVG）属于**股票交易活跃度类因子**，通过衡量股票在月度内的平均交易活跃度来反映投资者对该股票的关注程度。股票交易越活跃，说明市场参与者对该股票的买卖意愿越强，投资者对该股票的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。高关注度往往伴随着更多的非理性买入行为，在 A 股市场做多与做空不对称的背景下，这种非理性净买入会推动股价产生短期溢价，进而在未来引发价格反转。

数据：日度换手率

# 基于 PE 的分析师锚定效应偏差（CAF_EP）

**行为金融因子 —— 分析师预测锚定效应**

$$
\text{CAF_EP}_{i,t} = \frac{\text{FEP}_{i,t} - \text{IFEP}_t}{\text{IFEP}_t}
$$

其中，$\text{FEP}_{i,t}$ 为分析师在时间 $t$ 对股票 $i$ 的 PE 预测值所对应的盈利预测（即 PE 的倒数）：

$$
\text{FEP}_{i,t} = \frac{1}{\text{FPE}_{i,t}}
$$

各变量定义如下：

* $\text{FPE}_{i,t}$ 为分析师在时间 $t$ 对股票 $i$ 的 PE 指标的预测值；
* $\text{FEP}_{i,t}$ 为股票 $i$ 在时间 $t$ 的预期盈利水平（PE 的倒数），用于衡量估值水平的高低；
* $\text{IFEP}_t$ 为股票 $i$ 所属行业中所有成分股的 $\text{FPE}_{i,t}$ 的中位数（即行业中位数 PE 的倒数）。

## 因子说明

**锚定效应**指人们在做出判断时，会将最初接触到的信息（即“锚点”）作为参考依据，并在其基础上进行调整，但调整往往不充分，从而导致判断偏差。在分析师预测场景中，锚定效应表现为：分析师会以公司所在行业的估值水平（行业中位数 PE）作为定价的“锚点”。当某只股票的估值指标超出行业中位数时（即 $\text{CAF_EP}_{i,t} > 0$），分析师会倾向于做出**悲观预测**，将预测值向行业中位数靠拢；反之，当估值低于行业中位数时（即 $\text{CAF_EP}_{i,t} < 0$），分析师会倾向于做出**乐观预测**。财报公布后，实际财务指标将与分析师预测形成反向修正：被悲观预测的公司，其实际财务指标将高于预测值，股价表现更优；被乐观预测的公司，其实际财务指标将低于预测值，股价表现更差。因此，该因子可用于预测未来股价的走势方向。

数据：PE分析师预测值、行业分类数据、PE

# 基于异常收益的注意力溢出（SPILL）

**行为金融因子 —— 投资者注意力溢出效应**

首先，计算每月平均异常日收益率作为个股层面的注意力代理指标：

$$
\text{ATTN}_{i,t} = \frac{1}{m} \sum_{t}^{t-m+1} \left( r_{i,t} - \bar{r}_t \right)^2
$$

其中，$r_{i,t}$ 为股票 $i$ 在交易日 $t$ 的收益率，$\bar{r}_t$ 为交易日 $t$ 的全市场收益率，$m$ 为交易日窗口，一般取为月度。进一步，计算当期相近股票的注意力均值与个股注意力之差，即为注意力溢出程度：

$$
\text{SPILL}_{i,t} = \text{ATTN}_{i,t} - \overline{\text{ATTN}}_{g(i),t}
$$

其中，$\overline{\text{ATTN}}_{g(i),t}$ 为股票 $i$ 所属分组 $g(i)$ 内所有股票在 $t$ 期的注意力均值（不含股票 $i$ 自身）。

## 分组标准

相近股票的确定标准如下：

1. 先以中信一级行业作为分类标准，将全市场股票划分为不同行业；
2. 在每个行业内，按市值规模从大到小排序，以 **3:4:3** 的比例将股票分为**大市值、中市值、小市值**三组；
3. 同一行业同一市值组内的股票即被定义为“相近股票”。

## 因子逻辑说明

投资者注意力存在**溢出效应**：当某只股票受到高度关注时，投资者往往会同时注意到与其特征相似（同行业、同市值规模）的“邻居”股票，从而导致这些相近股票也获得一定的关注度提升，并享有注意力溢价。上述因子以异常收益（收益相对于市场的偏离程度）作为注意力的代理指标，通过计算个股注意力与其相近股票平均注意力之间的差值来衡量溢出程度。$\text{SPILL}_{i,t}$ 越大，表明个股当前受到的关注显著高于其同类股票，未来注意力向相近股票溢出的效应越显著，相近股票预期将取得正向超额收益。

数据：中信一级行业、日度市值、日度收盘价

# 基于换手率的注意力溢出（SPILL）

**行为金融因子 —— 投资者注意力溢出效应**

首先，计算每月日度换手率均值作为个股层面的注意力代理指标：

$$
\text{ATTN}_{i,t} = \frac{1}{m} \sum_{t=1}^{t-m+1} \text{TURN}_{i,t}
$$

其中，$\text{TURN}_{i,t}$ 为股票 $i$ 在交易日 $t$ 的换手率，$m$ 为交易日窗口，一般取为月度。

进一步，计算当期相近股票的注意力均值与个股注意力之差，即为注意力溢出程度：

$$
\text{SPILL}_{i,t} = \text{ATTN}_{i,t} - \overline{\text{ATTN}}_{g(i),t}
$$

其中，$\overline{\text{ATTN}}_{g(i),t}$ 为股票 $i$ 所属分组 $g(i)$ 内所有股票在 $t$ 期的注意力均值（不含股票 $i$ 自身）。

## 分组标准

相近股票的确定标准如下：

1. 先以**中信一级行业**作为分类标准，将全市场股票划分为不同行业；
2. 在每个行业内，按市值规模从大到小排序，以 **3:4:3** 的比例将股票分为**大市值、中市值、小市值**三组；
3. 同一行业同一市值组内的股票即被定义为“相近股票”。

## 因子说明

投资者注意力存在溢出效应，当一只股票关注度较高时，投资者也会注意到与其相似的邻居股票，导致这些邻居股票的未来收益也会有不错的收益，享有注意力溢价；上述因子以日均换手率为注意力代理指标，计算注意力差值，注意力差值越大，表明未来注意力溢出效应越显著，股票预期会取得正向超额。

# 前景价值（TK）

**行为金融因子 —— 投资者前景理论偏好**

① 以个股过去 $N$ 日收益率序列作为未来收益分布的预测。将负收益率从小到大排列为 $r_{-m}, \dots, r_{-1}$，正收益率从小到大排列为 $r_1, \dots, r_n$，加上零收益 $r_0$，每个收益出现的概率均为 $1/N$，其中 $N = m + n + 1$：

$$
\left( r_{-m}, \frac{1}{N}; \dots; r_{-1}, \frac{1}{N}; r_0, \frac{1}{N}; r_1, \frac{1}{N}; \dots; r_n, \frac{1}{N} \right)
$$

② 利用价值函数和概率权重函数计算前景价值 TK：

$$
\text{TK} = \sum_{i=-m}^{-1} v(r_i) \left[ w\left(\frac{-i+m+1}{N}\right) - w\left(\frac{-i+m}{N}\right) \right] + \sum_{i=1}^{n} v(r_i) \left[ w\left(\frac{n-i+1}{N}\right) - w\left(\frac{n-i}{N}\right) \right]
$$

更紧凑地，前景价值可表达为：

$$
\text{TK} = \sum_{i=-m}^{n} v(r_i) \cdot \pi_i
$$

其中 $\pi_i$ 为对应于收益率 $r_i$ 的概率权重。

③ 价值函数 $v(x)$ 和概率权重函数 $w^+(P)$、$w^-(P)$ 如下所示，函数参数取值为 $\alpha = 0.88$，$\lambda = 2.25$，$\gamma = 0.61$，$\delta = 0.69$：

$$
v(x) =\begin{cases}x^{\alpha}, & x \geq 0 \\-\lambda (-x)^{\alpha}, & x < 0\end{cases}
$$$$
w^+(P) = \frac{P^{\gamma}}{\left( P^{\gamma} + (1-P)^{\gamma} \right)^{1/\gamma}}
$$$$
w^-(P) = \frac{P^{\delta}}{\left( P^{\delta} + (1-P)^{\delta} \right)^{1/\delta}}
$$

其中，$w^+(P)$ 用于盈利概率的权重处理，$w^-(P)$ 用于亏损概率的权重处理。

## 因子说明

前景价值（TK）代表了投资者对每只个股的“综合打分”，其理论基础源于行为金融学中的**前景理论**（Prospect Theory）。根据前景理论，投资者在决策时并非基于客观的期望效用，而是基于对收益和损失的主观价值评估，同时对小概率事件存在过度权重、对大概率事件存在权重不足的倾向。TK 因子通过将个股的历史收益率分布代入价值函数和权重函数，量化了投资者对该股票的主观偏好程度。由于价值量 TK 高的股票在投资者眼中更具吸引力，投资者倾向于买入那些 TK 值较高的股票，导致这些股票被过度追捧、价格被高估；而 TK 值较低的股票则因缺乏关注而被低估。因此，股票的未来预期收益率与当前的前景价值 TK 呈**负相关**关系，TK 越高，未来收益越低；TK 越低，未来收益越高。

数据：日度收盘价

# 基于 PB 的前景价值 TK

**行为金融因子 —— 前景理论**

① 以个股过去 $N$ 期的估值变化率（PB 估值变化率）作为衡量公司价值的代理变量，构建未来价值变化率的概率分布。将负变化率从小到大排列为 $r_{pb,-m}, \dots, r_{pb,-1}$，正变化率从小到大排列为 $r_{pb,1}, \dots, r_{pb,n}$，加上零变化率 $r_{pb,0}$，每个变化率出现的概率均为 $1/N$，其中 $N = m + n + 1$：

$$
\left( r_{pb,-m}, \frac{1}{N}; \dots; r_{pb,-1}, \frac{1}{N}; r_{pb,0}, \frac{1}{N}; r_{pb,1}, \frac{1}{N}; \dots; r_{pb,n}, \frac{1}{N} \right)
$$

② 利用价值函数和概率权重函数计算前景价值 TK：

$$
\text{TK} = \sum_{i=-m}^{-1} v(r_{pb,i}) \left[ w^-\left(\frac{i+m+1}{N}\right) - w^-\left(\frac{i+m}{N}\right) \right] + \sum_{i=1}^{n} v(r_{pb,i}) \left[ w^+\left(\frac{n-i+1}{N}\right) - w^+\left(\frac{n-i}{N}\right) \right]
$$

③ 价值函数 $v(x)$ 和概率权重函数 $w^+(P)$、$w^-(P)$ 的参数取值与标准前景理论一致，取值为 $\alpha = 0.88$，$\lambda = 2.25$，$\gamma = 0.61$，$\delta = 0.69$：

$$
v(x) =\begin{cases}x^{\alpha}, & x \geq 0 \\-\lambda (-x)^{\alpha}, & x < 0\end{cases}
$$$$
w^+(P) = \frac{P^{\gamma}}{\left( P^{\gamma} + (1-P)^{\gamma} \right)^{1/\gamma}}
$$$$
w^-(P) = \frac{P^{\delta}}{\left( P^{\delta} + (1-P)^{\delta} \right)^{1/\delta}}
$$

其中，$w^+(P)$ 用于估值上升（盈利）概率的权重处理，$w^-(P)$ 用于估值下降（亏损）概率的权重处理。

## 因子说明

前景价值 TK 代表了投资者对每只个股估值水平的“综合打分”。与标准 TK 因子以历史收益率作为价值参考不同，**基于 PB 的前景价值 TK 采用 PB 估值变化率作为衡量公司价值的代理变量**。

其核心逻辑在于：投资者在决策时，会基于公司估值的历史变化形成参考点，高估值股票虽然当前价值量（TK）较高，但投资者倾向于认为其估值过高而选择卖出，导致卖出压力增大、股价被低估；相反，低估值股票因具备安全边际而受到投资者青睐，导致买入需求增加、股价被高估。

因此，在该框架下，**前景价值 TK 与股票未来预期收益呈正相关关系**：TK 值越高（对应高估值股票），未来收益反而可能更高（因超卖后的价格修复）；TK 值越低（对应低估值股票），未来收益反而可能更低（因超买后的价格回落）。

数据：PB估值

# 关注事件

**行为金融因子 —— 投资者注意力**

涨跌停事件数量和龙虎榜事件数量可作为投资者关注度的直接代理指标：

$$
\text{LimitEvent}_{i,t} = \sum_{d \in m} \left( \mathbb{1}_{\text{涨停},i,d} + \mathbb{1}_{\text{跌停},i,d} \right)
$$

其中：

* $\text{LimitEvent}_{i,t}$ 为股票 $i$ 在月度 $t$ 内发生涨停或跌停的总次数；
* $\mathbb{1}_{\text{涨停},i,d}$ 为指示函数，若股票 $i$ 在交易日 $d$ 涨停则取值为 1，否则为 0；
* $\mathbb{1}_{\text{跌停},i,d}$ 为指示函数，若股票 $i$ 在交易日 $d$ 跌停则取值为 1，否则为 0；
* $m$ 为月度交易日窗口。

$$
\text{DragonEvent}_{i,t} = \sum_{d \in m} \text{Dragon}_{i,d}
$$

其中：

* $\text{DragonEvent}_{i,t}$ 为股票 $i$ 在月度 $t$ 内登上龙虎榜的总次数；
* $\text{Dragon}_{i,d}$ 为股票 $i$ 在交易日 $d$ 登上龙虎榜的次数；
* $m$ 为月度交易日窗口。

## 应用方法

在实际投资组合构建中，可以每期买入上述涨跌停事件或龙虎榜事件数量排名前 50% 的股票构建投资组合，通过检验该组合的未来收益表现来评估关注事件类因子的选股能力。

## 因子说明

涨跌停股票和龙虎榜股票通常因其价格剧烈波动或受到监管机构公开披露而容易引起市场的广泛关注。这些特殊事件的发生，使得相关股票在短时间内暴露于大量投资者的视野中，成为市场关注的焦点。

因此，涨跌停事件数量和龙虎榜事件数量可以作为投资者关注度的有效代理指标。事件发生频率越高，表明该股票受市场关注的程度越高，后续可能引发更多的交易行为，并对股价产生相应的影响。

数据：日度涨跌停数据（日度收盘价、日度开盘价）、日度龙虎榜数据

# 基于涨跌停的注意力捕捉

**行为金融因子 —— 投资者注意力**

首先，将每日涨跌停股票数量除以当日交易的股票总数，得到每日全市场涨跌停比率，作为市场关注程度的代理指标：

$$
\text{LIMIT\_PROP}_d = \frac{n_{\text{upper},d} + n_{\text{lower},d}}{N_d}
$$

其中：

* $n_{\text{upper},d}$ 为交易日 $d$ 的涨停股票数量；
* $n_{\text{lower},d}$ 为交易日 $d$ 的跌停股票数量；
* $N_d$ 为交易日 $d$ 的交易股票总数。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{LIMIT\_PROP}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受涨跌停关注事件影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而引发更多的买盘压力，导致股价被高估。

上述因子以涨跌停比率作为市场关注度事件，其原因在于：涨跌停是市场上最引人注目的事件之一，涨跌停股票通常能够迅速吸引大量投资者的关注。该比率越高，说明市场上发生极端价格变动的股票越多，整体市场关注度也就越高。

数据：日度涨跌停数据（日度收盘价、日度开盘价）、日度可交易股票数量

# LSV 模型

**用于捕捉基金“抱团”羊群效应的行为金融模型**

LSV 模型的核心指标计算如下：

$$
\text{LSV}_{i,t} = \left| p_{i,t} - p_t \right| - \mathbb{E}\left| p_{i,t} - p_t \right|
$$

其中，第二项为在无羊群效应假设下的期望绝对偏差，其计算公式为：

$$
\mathbb{E}\left| p_{i,t} - p_t \right| = \text{AF}(i,t) = \sum_{k=1}^{N_{i,t}} \binom{N_{i,t}}{k} p_t^k (1 - p_t)^{N_{i,t}-k} \left| \frac{k}{N_{i,t}} - p_t \right|
$$

各基础变量定义如下：

$$
p_{i,t} = \frac{B(i,t)}{B(i,t) + S(i,t)}
$$$$
p_t = \frac{\sum_{i=1}^{N} p_{i,t}}{N}
$$$$
N_{i,t} = B(i,t) + S(i,t)
$$

其中：

* $B(i,t)$ 为在区间 $t$ 内买入股票 $i$ 的基金数量（即报告期持仓相较上期增加的基金数）；
* $S(i,t)$ 为在区间 $t$ 内卖出股票 $i$ 的基金数量（即报告期持仓相较上期减少的基金数）；
* $N_{i,t}$ 为持仓发生变动的基金总数（买入基金与卖出基金之和）；
* $p_{i,t}$ 为股票 $i$ 在区间 $t$ 内买入基金的占比；
* $p_t$ 为全市场所有股票在区间 $t$ 内买入基金占比的均值（跨股票平均值）；
* $N$ 为全市场股票总数。

## 计算示例

以基金年报或半年报持仓数据为例，若在报告期内有 300 只基金对股票 $i$ 进行了加仓操作，50 只基金进行了减仓操作，则：

$$
B(i,t) = 300,\quad S(i,t) = 50,\quad N_{i,t} = 350
$$$$
p_{i,t} = \frac{300}{350} \approx 0.857
$$

## 因子说明

LSV 模型从基金持股变动数据出发，通过比较某只股票实际被买入的比例与全市场平均买入比例的偏差程度，来判断该股票是否被基金集中买入或集中卖出，从而有效捕捉基金“抱团”现象。

当 $\text{LSV}_{i,t} > 0$ 且数值较大时，表明股票 $i$ 在当期存在显著的基金集体买入或集体卖出行为，即基金“抱团”现象明显。基金抱团是股票出现羊群效应的潜在驱动因子之一，抱团行为可能导致股价偏离基本面价值，并在未来出现价格反转。

数据：基金持仓数据

# SLSV 模型（对称 LSV 模型）

**行为金融因子 —— 羊群效应（基金买卖方向识别）**

SLSV 模型在 LSV 模型的基础上引入方向符号，以区分基金的买入从众和卖出从众行为：

$$
\text{SLSV}_{i,t} =\begin{cases}\text{LSV}_{i,t}, & p_{i,t} > p_t, \\-\text{LSV}_{i,t}, & p_{i,t} < p_t\end{cases}
$$

其中，$\text{LSV}_{i,t}$ 为 LSV 模型的核心指标，计算方式如下：

$$
\text{LSV}_{i,t} = \left| p_{i,t} - p_t \right| - \mathbb{E}\left| p_{i,t} - p_t \right|
$$

期望绝对偏差项在无羊群效应假设下计算：

$$
\mathbb{E}\left| p_{i,t} - p_t \right| = \text{AF}(i,t) = \sum_{k=1}^{N_{i,t}} \binom{N_{i,t}}{k} p_t^k (1 - p_t)^{N_{i,t}-k} \left| \frac{k}{N_{i,t}} - p_t \right|
$$

各基础变量定义如下：

$$
p_{i,t} = \frac{B(i,t)}{B(i,t) + S(i,t)}
$$$$
p_t = \frac{\sum_{i=1}^{N} p_{i,t}}{N}
$$$$
N_{i,t} = B(i,t) + S(i,t)
$$

其中：

* $B(i,t)$ 为在区间 $t$ 内买入股票 $i$ 的基金数量（即报告期持仓相较上期增加的基金数）；
* $S(i,t)$ 为在区间 $t$ 内卖出股票 $i$ 的基金数量（即报告期持仓相较上期减少的基金数）；
* $N_{i,t}$ 为持仓发生变动的基金总数（买入基金与卖出基金之和）；
* $p_{i,t}$ 为股票 $i$ 在区间 $t$ 内买入基金的占比；
* $p_t$ 为全市场所有股票在区间 $t$ 内买入基金占比的均值（跨股票平均值）；
* $N$ 为全市场股票总数。

## 计算示例

以基金年报或半年报持仓数据为例，若在报告期内有 300 只基金对股票 $i$ 进行了加仓操作，50 只基金进行了减仓操作，则：

$$
B(i,t) = 300,\quad S(i,t) = 50,\quad N_{i,t} = 350
$$$$
p_{i,t} = \frac{300}{350} \approx 0.857
$$

## 因子说明

LSV 模型从基金持股变动数据出发，通过比较某只股票实际被买入的比例与全市场平均买入比例的偏差程度，来判断该股票是否存在基金集中交易行为，从而捕捉基金“抱团”现象。

在此基础上，**SLSV 模型（对称 LSV）进一步考虑了投资者的买卖方向**，对 LSV 指标进行符号化处理：

* 当 $\text{SLSV}_{i,t} > 0$ 时（即 $p_{i,t} > p_t$），表明基金对该股票的买入比例高于市场平均水平，对应**买入从众效应**（集体加仓）；
* 当 $\text{SLSV}_{i,t} < 0$ 时（即 $p_{i,t} < p_t$），表明基金对该股票的买入比例低于市场平均水平（卖出比例更高），对应**卖出从众效应**（集体减仓）。

SLSV 指标的正负方向能够有效区分基金是“集体买入”还是“集体卖出”某只股票，为研究羊群效应的方向性差异提供了更丰富的信息。

数据：基金持仓数据

In [4]:
import sys

!{sys.executable} -m pip install --user --upgrade --no-cache-dir -i https://pypi.org/simple "rqdatac==3.4.2" "rqdatac-fund==1.0.44"

     |████████████████████████████████| 582 kB 16 kB/s eta 0:00:01
Requirement already up-to-date: rqdatac-fund==1.0.44 in /home/rice/notebook/.packages/lib/python3.6/site-packages (1.0.44)
  Attempting uninstall: rqdatac
    Found existing installation: rqdatac 3.0.15
    Uninstalling rqdatac-3.0.15:
      Successfully uninstalled rqdatac-3.0.15


## 数据获取（付费接口）

In [ ]:
# 基金持仓数据.py

# 🚨基于异常收益的注意力捕捉

**行为金融因子 —— 投资者注意力**

对每个行业，分别统计每日行业内成分股中收益率绝对值高于 7% 的股票数量占比，作为市场关注程度的代理指标：

$$
\text{indus_abnd}_d = \frac{n_{\ge 7\%, d}}{N_d}
$$

其中，$n_{\ge 7\%, d}$ 为行业成分股中当日收益率绝对值达到或超过 7% 的股票数量，$N_d$ 为该行业当日交易股票总数。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{indus_abnd}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \epsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受投资者注意力影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\epsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而引发更多的买盘压力，导致股价被高估。

上述因子以异常收益占比（收益率绝对值超过 7% 的股票比例）作为市场关注度事件，其原因在于：极端日收益是衡量关注度的一个重要指标，出现过高或过低的收益都代表着该股票或行业受到了投资者的广泛关注。因此，该指标能够有效捕捉投资者注意力的集中程度，并进一步用于解释个股收益的横截面差异。

数据：日度收盘价、Fama-French因子（市值、无风险收益率、账面市值比（净资产/总市值））、行业分类数据

# 🚨基于市场波动率的注意力捕捉

**行为金融因子 —— 投资者注意力**

对每个行业，分别统计每日行业内成分股收益率的横截面标准差，作为市场关注程度的代理指标：

$$
\text{indus_vol}_d = \text{std}\left( r_{i,d} \right), \quad i \in \text{行业内成分股}
$$

其中，$r_{i,d}$ 为行业成分股 $i$ 在交易日 $d$ 的收益率，$\text{indus_vol}_d$ 为行业内所有成分股当日收益率的横截面标准差，用于衡量该行业当日的收益分歧度和交易活跃程度。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{indus_vol}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受行业波动率变化影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{indus_vol}_d$ 为交易日 $d$ 行业内成分股收益率的横截面标准差；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而获取超额收益。

上述因子以行业内收益率的横截面标准差（即行业波动率）作为市场关注度事件，其原因在于：行业内成分股收益的分化程度反映了市场对该行业内部不同股票的分歧和关注。当行业波动率较高时，表明该行业内部收益差异较大，市场交易活跃，关注度较高；反之，则表明行业关注度较低。因此，该指标能够有效捕捉市场注意力的变化，并用于解释个股收益的横截面差异。

数据：日度收盘价、行业分类数据、Fama‑French 三因子

# 🚨基于沪深总成交额的注意力捕捉

**行为金融因子 —— 投资者注意力**

首先，统计每日沪深两市的总成交额，作为市场关注程度的代理指标：

$$
\text{mkt_vol}_d = \text{Vol}_{\text{SH},d} + \text{Vol}_{\text{SZ},d}
$$

其中，$\text{Vol}_{\text{SH},d}$ 为上海证券交易所所有股票在交易日 $d$ 的总成交额，$\text{Vol}_{\text{SZ},d}$ 为深圳证券交易所所有股票在交易日 $d$ 的总成交额。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{mkt_vol}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受全市场成交额变化影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{mkt_vol}_d$ 为交易日 $d$ 的沪深两市总成交额；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而导致其股价波动性增大，买盘压力增加，股价被高估。

上述因子以沪深两市总成交额作为市场关注度事件，其原因在于：市场总成交额是反映全市场交易活跃度和投资者参与热情的最直接指标之一。成交额越高，说明市场整体关注度越高，资金流动越活跃，处于该市场环境下的个股受到的关注程度也随之提升。

数据：股票分类数据（上海交易所、深圳交易所）、日度成交额、日度收盘价、Fama‑French 三因子

# 基于换手率的注意力捕捉

**行为金融因子 —— 投资者注意力**

对每个行业，分别统计每日行业内成分股换手率的横截面标准差，作为市场关注程度的代理指标：

$$
\text{indus\_turn}_d = \text{std}\left( \text{turnover}_{i,d} \right), \quad i \in \text{行业内成分股}
$$

其中，$\text{turnover}_{i,d}$ 为股票 $i$ 在交易日 $d$ 的换手率，$\text{indus\_turn}_d$ 为行业内所有成分股当日换手率的标准差，用于衡量该行业当日的交易活跃度和关注度分歧程度。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{indus\_turn}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受投资者注意力影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而引发更多的买盘压力，导致股价被高估。上述因子以行业内换手率的横截面标准差作为市场关注度事件，其原因在于：换手率是衡量投资者交易热度的常用指标，能够有效反映市场对特定行业或个股的关注水平。换手率越高，表明交易越活跃，投资者关注度也越高。

数据：日度换手率、日度收盘价、Fama-French因子（市值、无风险收益率、账面市值比（净资产/总市值））